# WALKBACK step 1: fetch the raw data
<!-- AI-assisted: written with Claude (Opus 5.5), reviewed by the team. -->

Press **Runtime > Run all**. This notebook:
1. cuts our 1.07 x 1.36 km site out of the USGS 1 m Gale elevation model (reads only our window over the network, not the whole 3.6 GB file),
2. tries to cut the matching colour image from the USGS HiRISE colour mosaic (optional),
3. downloads Curiosity's waypoints, traverse and REMS weather records from NASA,
4. zips everything into `walkback_raw.zip` and downloads it to your computer.

Sources are listed in `docs/SPEC.md` section 5.

In [ ]:
!pip -q install rasterio

In [ ]:
import os, json, zipfile, time
import numpy as np, requests, rasterio
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform
from rasterio.warp import transform_bounds

# Make GDAL read remote files efficiently (only the bytes we need).
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif"

OUT = "/content/walkback_raw"
os.makedirs(OUT, exist_ok=True)

# Site bounds in degrees, east-positive longitude (docs/SPEC.md section 2)
W, E, S, N = 137.3740, 137.3920, -4.8170, -4.7940
# Mars as a sphere of radius 3396190 m, in plain longitude/latitude
MARS_LONLAT = "+proj=longlat +a=3396190 +b=3396190 +no_defs"

DEM_URL   = "https://planetarymaps.usgs.gov/mosaic/Mars/MSL/MSL_Gale_DEM_Mosaic_1m_v3.tif"
ORTHO_URL = "https://asc-pds-services.s3.us-west-2.amazonaws.com/mosaic/Mars/MSL/MSL_Gale_HiRISE-LRGB_78quads_sharp_cog.tif"
print("ready")

In [ ]:
# 1. Elevation crop
t0 = time.time()
with rasterio.open("/vsicurl/" + DEM_URL) as src:
    print("DEM size:", src.width, "x", src.height, "| bands:", src.count, "| dtype:", src.dtypes[0])
    print("DEM CRS:", src.crs.to_string())
    print("DEM pixel size (m):", src.res, "| nodata:", src.nodata)
    xmin, ymin, xmax, ymax = transform_bounds(MARS_LONLAT, src.crs, W, S, E, N)
    win = from_bounds(xmin, ymin, xmax, ymax, src.transform).round_offsets().round_lengths()
    print("window:", win)
    dem = src.read(1, window=win).astype("float32")
    if src.nodata is not None:
        dem[dem == src.nodata] = np.nan
    profile = dict(driver="GTiff", height=dem.shape[0], width=dem.shape[1], count=1,
                   dtype="float32", crs=src.crs, transform=window_transform(win, src.transform),
                   compress="deflate", predictor=3, nodata=np.nan)
with rasterio.open(f"{OUT}/gale_dem_crop.tif", "w", **profile) as dst:
    dst.write(dem, 1)
print("crop shape (rows, cols):", dem.shape)
print("elevation min / max (m):", float(np.nanmin(dem)), float(np.nanmax(dem)), "| empty cells:", int(np.isnan(dem).sum()))
print("took %.0f s" % (time.time() - t0))

In [ ]:
# 2. Colour image crop (optional: skipped if anything goes wrong)
try:
    with rasterio.open("/vsicurl/" + ORTHO_URL) as src:
        print("ORTHO size:", src.width, "x", src.height, "| bands:", src.count, "| dtype:", src.dtypes[0])
        print("ORTHO CRS:", src.crs.to_string(), "| pixel size:", src.res)
        print("ORTHO overviews:", src.overviews(1))
        xmin, ymin, xmax, ymax = transform_bounds(MARS_LONLAT, src.crs, W, S, E, N)
        win = from_bounds(xmin, ymin, xmax, ymax, src.transform).round_offsets().round_lengths()
        scale = max(win.width, win.height) / 2200.0
        out_h, out_w = int(win.height / max(scale, 1)), int(win.width / max(scale, 1))
        bands = list(range(1, min(src.count, 3) + 1))
        img = src.read(bands, window=win, out_shape=(len(bands), out_h, out_w))
    from PIL import Image
    arr = np.moveaxis(img, 0, -1)
    if arr.dtype != np.uint8:
        lo, hi = np.percentile(arr[arr > 0], [1, 99])
        arr = np.clip((arr - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)
    if arr.shape[-1] == 1:
        arr = arr[..., 0]
    Image.fromarray(arr).save(f"{OUT}/ortho.jpg", quality=85)
    json.dump({"window_px": [win.col_off, win.row_off, win.width, win.height],
               "saved_px": [out_w, out_h], "crs": src.crs.to_string()},
              open(f"{OUT}/ortho_info.json", "w"))
    print("ortho saved:", out_w, "x", out_h)
except Exception as err:
    print("Colour image skipped:", repr(err))

In [ ]:
# 3. Curiosity waypoints, traverse and REMS weather from NASA
files = {
    "MSL_waypoints.json": "https://mars.nasa.gov/mmgis-maps/MSL/Layers/json/MSL_waypoints.json",
    "MSL_traverse.json":  "https://mars.nasa.gov/mmgis-maps/MSL/Layers/json/MSL_traverse.json",
    "rems_weather.json":  "https://mars.nasa.gov/rss/api/?feed=weather&category=msl&feedtype=json",
}
for name, url in files.items():
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    open(f"{OUT}/{name}", "wb").write(r.content)
    print(name, len(r.content) // 1024, "KB")

In [ ]:
# 4. Zip and download
zip_path = "/content/walkback_raw.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir(OUT)):
        z.write(f"{OUT}/{name}", name)
print("zip size: %.1f MB" % (os.path.getsize(zip_path) / 1e6))
from google.colab import files
files.download(zip_path)
print("DONE: walkback_raw.zip is downloading")